# FinScore AI — Credit Score Classification ML Project

*Epsilon AI | Course Final Project | Developed by Ali Hamed*

---

## Project Overview

**FinScore AI** is an end-to-end Machine Learning classification system and interactive web application built with **Streamlit**. Designed for modern banking and financial risk assessment, the platform predicts customer creditworthiness across three tiers (**Good**, **Standard**, and **Poor**) based on global **FICO** credit scoring pillars—powered by a tuned **XGBoost** pipeline, automated feature engineering, and dynamic financial advisory insights.

---

## Dataset Source & Documentation

* **Source Datacard:** *Credit Score Classification Dataset* on Kaggle.
* **Description:** The dataset comprises **100,000** customer financial records across **28** raw features capturing income metrics, outstanding debt, credit utilization, payment delays, and credit history length, providing a comprehensive foundation for credit risk modeling.

---

## Citation & License

* **Dataset Attribution:** Rohan Paris, *Credit Score Classification Dataset*, Kaggle.
* **Data License:** CC0: Public Domain (subject to Kaggle's standard platform data use terms and repository sharing permissions for academic research and ML benchmarking).

---

## Packages & Libraries Setup


===============================================================================================================================================================================================

# Import And Reading Data

In [ ]:
import numpy as np 
import pandas as pd 
import plotly.express as px


In [ ]:
df = pd.read_csv("Credit_Score.csv")
df

# Step2: Data Understanding

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
(df.isnull().sum() / len(df))*100

In [ ]:
df.duplicated().sum()

In [ ]:
df.describe()

In [ ]:
df.columns

In [ ]:
df['Credit_Score'].value_counts()

# Data Cleaning

In [ ]:
df_clean = df.drop(["ID" ,"Name" ,"SSN"] ,axis=1)

In [ ]:
df_clean.head()

In [ ]:
def info_data(col):
    clean_col = col.strip()
    data = df_clean[clean_col]

    print(f"Column: {clean_col}")

    print("=" * 100)
    
    print("dtype:", data.dtype)

    print("=" * 100)

    print("unique values:", data.nunique())

    print("=" * 100)

    print("missing (NaN):", data.isnull().sum())

    print("=" * 100)

    print(data.value_counts(dropna=False).head(10))

    



In [ ]:
def fix_by_group(target_col, method="auto", group_col="Customer_ID"):
    # auto: رقمي → median ، نصي → mode
    if method == "auto":
        method = "median" if pd.api.types.is_numeric_dtype(df_clean[target_col]) else "mode"

    def _mode(x):
        m = x.mode()
        return m.iloc[0] if not m.empty else np.nan

    func = _mode if method == "mode" else "median"
    df_clean[target_col] = df_clean.groupby(group_col)[target_col].transform(func)

In [ ]:
# الفانكشن دي بتعوّض القيم الفاضية (NaN) في أي عمود من نفس العميل.
# بجيب القيمة المناسبة لكل عميل (median للأرقام أو mode للنصوص)، وبحطها مكان الفاضي بس.
# القيم السليمة اللي في الشهور التانية مش بلمسها، فالتغيير الحقيقي من شهر لشهر بيفضل زي ما هو.
# لو عميل كل شهوره فاضية، باخد القيمة العامة للعمود كله كحل أخير.
def fill_nan(col, method="median"):
    if method == "mode":
        g = df_clean.groupby("Customer_ID")[col].transform(
            lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
        df_clean[col] = df_clean[col].fillna(g).fillna(df_clean[col].mode().iloc[0])
    else:
        g = df_clean.groupby("Customer_ID")[col].transform("median")
        df_clean[col] = df_clean[col].fillna(g).fillna(df_clean[col].median())

In [ ]:
info_data('Customer_ID')

In [ ]:
df_clean[['Customer_ID']].head(20)

In [ ]:
8 * 12500

In [ ]:
info_data('Age')

In [ ]:
df['Age'].head(10000).unique()

In [ ]:
df_clean['Age'] = df_clean['Age'].astype(str).str.strip().str.replace("_","" , regex=False)

In [ ]:
df_clean['Age']

In [ ]:
df_clean['Age'] = df_clean['Age'].astype(int)

In [ ]:
# تم تنظيف السن استنادا لعمر الدولي للسماح للقروض الحد الادني 18 و الحد الاقصي 70 عام

df_clean.loc[(df_clean['Age'] < 18) | (df_clean['Age'] > 70), 'Age'] = np.nan

In [ ]:
df_clean["Age"].isnull()

In [ ]:
#    groupby لتنفيذ علي الملء بستخدام  transform التعويض الأول من شهور العميل نفسه استخدمنا ب(Customer_ID) لو عنده شهور سليمة
df_clean['Age'] = df_clean.groupby('Customer_ID')['Age'].transform(lambda x: x.fillna(x.median()))

# fillna بنسبه للعملاءالي كل شهورهم بايظة نعوضهم بستخدام ب سن العملاء الي في نفس الوظيفه بتعته بستخدام 
df_clean['Age'] = df_clean['Age'].fillna(df_clean.groupby('Occupation')['Age'].transform('median'))

# ضمان علشان لو في اي حله حصل انها متمش اضافه له سن بنخذ من المتوسط للعمود كله و هي نسبه شبه معدومه
df_clean['Age'] = df_clean['Age'].fillna(df_clean['Age'].median())


In [ ]:
df_clean['Age'] = df_clean['Age'].astype(int)
df_clean['Age']

In [ ]:
nun_age = df_clean['Age'].isna().sum()
Customer_nun_age = nun_age / 8
Customer_nun_age

In [ ]:
info_data('Occupation')

In [ ]:
df_clean[df_clean["Occupation"]=='_______']

In [ ]:
df_clean.loc[(df_clean['Occupation'] == '_______','Occupation') ] = np.nan

In [ ]:
fix_by_group('Occupation', method='mode')

In [ ]:
df_clean['Occupation'].value_counts()

In [ ]:
info_data('Annual_Income')

In [ ]:
df_clean['Annual_Income'] = df_clean['Annual_Income'].astype(str).str.strip().str.replace("_","" , regex=False)

In [ ]:
df_clean['Annual_Income'] = df_clean['Annual_Income'].astype(float)

In [ ]:
df_clean['Annual_Income'].describe()

In [ ]:
fix_by_group('Annual_Income', method='mode')

In [ ]:
df_clean['Annual_Income'].describe()

In [ ]:
df_clean.groupby("Customer_ID")["Annual_Income"].nunique().value_counts()

In [ ]:
df_clean.head()

In [ ]:
info_data('Monthly_Inhand_Salary')

In [ ]:
fix_by_group('Monthly_Inhand_Salary', method= 'mode')

In [ ]:
df_clean.groupby("Customer_ID")["Monthly_Inhand_Salary"].nunique().value_counts()

In [ ]:
df_clean['Monthly_Inhand_Salary'].describe()

In [ ]:
info_data('Num_Bank_Accounts')

In [ ]:
df_clean['Num_Bank_Accounts'].describe()

In [ ]:
df_clean.loc[(df_clean['Num_Bank_Accounts'] < 0) , 'Num_Bank_Accounts'] = np.nan

In [ ]:
fix_by_group('Num_Bank_Accounts',method='mode')

In [ ]:
df_clean['Num_Bank_Accounts'].describe()

In [ ]:
df_clean.groupby("Customer_ID")["Num_Bank_Accounts"].nunique().value_counts()

In [ ]:
info_data('Num_Credit_Card')

In [ ]:
df_clean['Num_Credit_Card'].describe()

In [ ]:
fix_by_group('Num_Credit_Card', method=('mode'))

In [ ]:
df_clean['Num_Credit_Card'].describe()

In [ ]:
df_clean.groupby("Customer_ID")["Num_Credit_Card"].nunique().value_counts()

In [ ]:
info_data('Interest_Rate')

In [ ]:
fix_by_group('Interest_Rate', method='mode')

In [ ]:
df_clean['Interest_Rate'].describe()

In [ ]:
df_clean.groupby("Customer_ID")['Interest_Rate'].nunique().value_counts()

In [ ]:
info_data('Num_of_Loan')

In [ ]:
df_clean['Num_of_Loan'] = df_clean['Num_of_Loan'].astype(str).str.strip().str.replace("_","" , regex=False)

In [ ]:
df_clean['Num_of_Loan'] = df_clean['Num_of_Loan'].astype(int)

In [ ]:
df_clean.loc[(df_clean['Num_of_Loan'] < 0) , 'Num_of_Loan'] = np.nan

In [ ]:
fix_by_group('Num_of_Loan', method='mode')

In [ ]:
df_clean['Num_of_Loan'].describe()

In [ ]:
df_clean.groupby("Customer_ID")["Num_of_Loan"].nunique().value_counts()

In [ ]:
df_clean.columns

In [ ]:
info_data('Type_of_Loan')

In [ ]:
# (No Loan) تم التاكيد من ان القيم المفقوده كانت مع القيم العملاء اللذين ليس لهم قروض اذا يتم التعويض ب 
df_clean[df_clean["Num_of_Loan"] == 0]["Type_of_Loan"].isnull().value_counts()

df_clean[df_clean["Type_of_Loan"].isnull()]["Num_of_Loan"].value_counts()

In [ ]:
df_clean["Type_of_Loan"] = df_clean["Type_of_Loan"].fillna("No Loan")


In [ ]:
info_data('Type_of_Loan')

In [ ]:
df_clean.groupby("Customer_ID")["Num_of_Loan"].nunique().value_counts()

In [ ]:
info_data('Delay_from_due_date')

In [ ]:
df_clean['Delay_from_due_date'].describe()

In [ ]:
df_clean['Delay_from_due_date'].describe()

In [ ]:
df_clean.groupby("Customer_ID")["Delay_from_due_date"].nunique().value_counts()

In [ ]:
df_clean['Num_of_Delayed_Payment'] = df['Num_of_Delayed_Payment']

In [ ]:
info_data('Num_of_Delayed_Payment')

In [ ]:
df_clean['Num_of_Delayed_Payment'].unique()

In [ ]:
df_clean['Num_of_Delayed_Payment'] = df_clean['Num_of_Delayed_Payment'].astype(str).str.strip().str.replace("_","" , regex=False)

In [ ]:
df_clean['Num_of_Delayed_Payment'] = df_clean['Num_of_Delayed_Payment'].astype(float)

In [ ]:
df_clean.loc[((df_clean['Num_of_Delayed_Payment'] < 0) | (df_clean['Num_of_Delayed_Payment'] > 30) ) , 'Num_of_Delayed_Payment'] = np.nan

In [ ]:
fill_nan('Num_of_Delayed_Payment', method='mode')

In [ ]:
df_clean.groupby("Customer_ID")["Num_of_Delayed_Payment"].nunique().value_counts()

In [ ]:
df_clean['Num_of_Delayed_Payment'].describe()

In [ ]:
df_clean.columns

In [ ]:
info_data('Changed_Credit_Limit')

In [ ]:

s = df["Changed_Credit_Limit"].astype(str).str.strip().replace("_", np.nan)
df_clean["Changed_Credit_Limit"] = pd.to_numeric(s.str.replace("_", "", regex=False), errors="coerce")
fill_nan("Changed_Credit_Limit")

In [ ]:
df_clean['Changed_Credit_Limit'] = df_clean['Changed_Credit_Limit'].astype(float)

In [ ]:
df_clean['Changed_Credit_Limit'].describe()

In [ ]:
df_clean.groupby("Customer_ID")["Changed_Credit_Limit"].nunique().value_counts()

In [ ]:
info_data('Num_Credit_Inquiries')

In [ ]:
df_clean['Num_Credit_Inquiries'].unique()

In [ ]:
fix_by_group('Num_Credit_Inquiries', method='mode')

In [ ]:
df_clean['Num_Credit_Inquiries'].describe()

In [ ]:
df_clean.groupby("Customer_ID")["Num_Credit_Inquiries"].nunique().value_counts()

In [ ]:
info_data('Credit_Mix')

In [ ]:
df_clean['Credit_Mix'].unique()

In [ ]:
df_clean.loc[(df_clean['Credit_Mix'] == '_','Credit_Mix') ] = np.nan

In [ ]:
fix_by_group('Credit_Mix', method='mode')

In [ ]:
df_clean['Credit_Mix'].unique()

In [ ]:
df_clean.groupby("Customer_ID")["Credit_Mix"].nunique().value_counts()

In [ ]:
info_data('Outstanding_Debt')

In [ ]:
df_clean['Outstanding_Debt'].unique()

In [ ]:
df_clean['Outstanding_Debt'] = df_clean['Outstanding_Debt'].astype(str).str.strip().str.replace("_","" , regex=False)

In [ ]:
df_clean['Outstanding_Debt'] = df_clean['Outstanding_Debt'].astype(float)

In [ ]:
fix_by_group('Outstanding_Debt', method='mode')

In [ ]:
df_clean['Outstanding_Debt'].describe()

In [ ]:
df_clean.groupby("Customer_ID")["Outstanding_Debt"].nunique().value_counts()

In [ ]:
info_data('Credit_Utilization_Ratio')

In [ ]:
df_clean['Credit_Utilization_Ratio'].describe()

In [ ]:
df_clean.columns

In [ ]:
info_data('Credit_History_Age')

In [ ]:
df_clean['Credit_History_Age'].unique()

In [ ]:
# العمود نص ("22 Years and 1 Months") فحوّلته لعدد شهور، وده عداد بيزيد شهر كل صف
month_map = {m: i for i, m in enumerate(["January", "February", "March", "April", "May", "June", "July", "August"])}
mnum = df_clean["Month"].map(month_map)
ym = df["Credit_History_Age"].str.extract(r"(\d+) Years and (\d+) Months").astype(float)
months = ym[0] * 12 + ym[1]

# بحسب نقطة بداية كل عميل (الشهور ناقص رقم الشهر) وأملا الفاضي بيها زائد رقم شهره
base = (months - mnum).groupby(df_clean["Customer_ID"]).transform("median")
df_clean["Credit_History_Age"] = months.fillna(base + mnum).fillna(months.median())

In [ ]:
df_clean['Credit_History_Age'].unique()

In [ ]:
df_clean.groupby("Customer_ID")["Credit_History_Age"].nunique().value_counts()

In [ ]:
info_data('Payment_of_Min_Amount')

In [ ]:
# تم تبديل  nm الي nan 
df_clean["Payment_of_Min_Amount"] = df_clean["Payment_of_Min_Amount"].replace("NM", np.nan)


In [ ]:
fix_by_group('Payment_of_Min_Amount', method='mode')

In [ ]:

df_clean["Payment_of_Min_Amount"].unique()

In [ ]:
df_clean.groupby("Customer_ID")["Payment_of_Min_Amount"].nunique().value_counts()

In [ ]:
info_data('Total_EMI_per_month')

In [ ]:
# قيم Total_EMI_per_month الضخمة (حتى 82 ألف) كانت أخطاء تسجيل، واتصلحت من نفس العميل بالـ mode، والـ max بقى 1779
df_clean[df_clean["Total_EMI_per_month"].between(1000, 50000)].shape[0]

In [ ]:
fix_by_group('Total_EMI_per_month', method='mode')

In [ ]:
df_clean['Total_EMI_per_month'].describe()

In [ ]:
df_clean.groupby("Customer_ID")["Total_EMI_per_month"].nunique().value_counts()

In [ ]:
info_data('Amount_invested_monthly')

In [ ]:
# "__10000__" قيمة وهمية، حوّلتها NaN قبل ما أشيل أي رموز، وبعدها عوّضت من نفس العميل
s = df["Amount_invested_monthly"].replace("__10000__", np.nan)
df_clean["Amount_invested_monthly"] = pd.to_numeric(s, errors="coerce")
fill_nan("Amount_invested_monthly")

In [ ]:
df_clean['Amount_invested_monthly'].describe()

In [ ]:
df_clean.groupby("Customer_ID")["Amount_invested_monthly"].nunique().value_counts()

In [ ]:
df_clean.columns

In [ ]:
info_data('Payment_Behaviour')

In [ ]:
df_clean['Payment_Behaviour'].unique()

In [ ]:
# "!@9#%8" رمز غلط، حوّلته NaN وعوّضت بـ mode نفس العميل (العمود نصي)
df_clean["Payment_Behaviour"] = df["Payment_Behaviour"].replace("!@9#%8", np.nan)
fill_nan("Payment_Behaviour", method="mode")

In [ ]:
df_clean['Payment_Behaviour'].unique()

In [ ]:
df_clean.groupby("Customer_ID")["Payment_Behaviour"].nunique().value_counts()

In [ ]:
info_data('Monthly_Balance')

In [ ]:
df_clean["Monthly_Balance"] = df_clean["Monthly_Balance"].replace("__-333333333333333333333333333__", np.nan)


In [ ]:
df_clean["Monthly_Balance"] = df_clean["Monthly_Balance"].astype(float)

In [ ]:
med = df_clean.groupby("Customer_ID")["Monthly_Balance"].transform("median")
df_clean["Monthly_Balance"] = df_clean["Monthly_Balance"].fillna(med)

In [ ]:
df_clean["Monthly_Balance"].describe()

In [ ]:
df_clean.groupby("Customer_ID")["Monthly_Balance"].nunique().value_counts()

In [ ]:
df_clean.columns

In [ ]:
info_data('Credit_Score')

In [ ]:
df_clean.groupby("Customer_ID")["Credit_Score"].nunique().value_counts()

In [ ]:
df_clean.isnull().sum()

==================================================================================================================================

<div dir="rtl">

## مرحلة التحليل الاستكشافي للبيانات (EDA)

سأبدأ في هذا القسم تحليل أعمدة البيانات بناءً على **نموذج FICO العالمي (Fair Isaac Corporation)**، وهو المعيار المالي الأشهر والأكثر اعتماداً في البنوك لتقييم الجدارة الائتمانية للعملاء؛ حيث يرتكز تقييمه على **5 عوامل رئيسية**: تاريخ السداد (**35%**)، حجم الديون المستغلة (**30%**)، طول تاريخ الائتمان (**15%**)، أنواع الائتمان (**10%**)، وطلبات الائتمان الجديدة (**10%**).

</div>


=============================================================================================================

In [ ]:
numaric_df = df_clean.select_dtypes(include=[np.number])

In [ ]:
numaric_df.corr()

In [ ]:
px.pie(df_clean, names="Credit_Score", title="Credit Score Distribution")

<div dir="rtl">

* يتضح من الرسم البياني الدائري (**Pie Chart**) لتوزيع التصنيف الائتماني (**Credit_Score**) أن الفئة المتوسطة (**Standard**) تمثل النسبة الأكبر من العملاء بواقع **53.2%**، تليها فئة التصنيف المنخفض (**Poor**) بنسبة **29%**، بينما تمثل فئة التصنيف الجيد (**Good**) النسبة الأقل بواقع **17.8%**؛ مما يشير إلى وجود عدم توازن واضح بين الفئات (**Class Imbalance**) في البيانات.

</div>


In [ ]:
px.histogram(df_clean, x="Monthly_Inhand_Salary", title="Monthly Inhand Salary Distribution")

<div dir="rtl">

* يتضح من الرسم البياني لتوزيع صافي الراتب الشهري (**Monthly_Inhand_Salary**) وجود التواء يميني واضح (**Right Skewness**)؛ حيث تتركز النسبة الأكبر من العملاء في نطاق الرواتب المنخفضة إلى المتوسطة ما بين **1,000 و4,000 دولار** شهرياً (وتبلغ الذروة حول **1,500 دولار**)، بينما تتناقص أعداد العملاء تدريجياً مع ارتفاع الراتب ليصل التوزيع حتى **15,000 دولار** شهرياً.

</div>


In [ ]:
px.box(df_clean, y="Monthly_Inhand_Salary", x="Credit_Score", title="Box Plot of Monthly Inhand Salary by Credit Score")

<div dir="rtl">

* يتضح من الرسم البياني للصندوق (**Box Plot**) وجود علاقة طردية بين ارتفاع صافي الراتب الشهري (**Monthly_Inhand_Salary**) ومستوى التصنيف الائتماني (**Credit Score**)؛ حيث يبلغ **الوسيط (Median)** لفئة التصنيف الجيد (**Good**) نحو **3,800 دولار** شهرياً (ويتركز معظمهم بين **2,400 و8,000 دولار**)، تليها فئة (**Standard**) بوسيط يبلغ نحو **3,000 دولار**، بينما تسجل فئة التصنيف المنخفض (**Poor**) أقل وسيط بنحو **2,600 دولار** شهرياً.
* كما يُلاحظ وجود قيم متطرفة مرتفعة (**Outliers**) في فئتي (**Standard**) و(**Poor**) لعملاء تتجاوز رواتبهم **10,000 دولار** شهرياً؛ مما يؤكد أن ارتفاع الدخل وحده لا يضمن تصنيفاً ائتمانياً مرتفعاً دون الالتزام بالسداد وإدارة الديون بكفاءة.

</div>


In [ ]:
px.histogram(df_clean, x="Annual_Income", title="Distribution of Annual Income")

<div dir="rtl">

* يتضح من الرسم البياني لتوزيع الدخل السنوي (**Annual_Income**) وجود التواء يميني واضح (**Right Skewness**)؛ حيث تتركز النسبة الأكبر من العملاء في نطاق الدخول المنخفضة إلى المتوسطة ما بين **15,000 و50,000 دولار** سنوياً (وتبلغ الذروة حول **20,000 دولار**)، بينما تتناقص أعداد العملاء تدريجياً مع ارتفاع الدخل ليصل التوزيع حتى **180,000 دولار** سنوياً، وهو ما يتطابق تماماً مع نمط توزيع صافي الراتب الشهري (**Monthly_Inhand_Salary**).

</div>


In [ ]:
px.box(df_clean, y='Annual_Income',x= 'Credit_Score' ,title="Box Plot of Annual Income by Credit Score")

<div dir="rtl">

* يتضح من الرسم البياني للصندوق (**Box Plot**) تطابق سلوك الدخل السنوي (**Annual_Income**) تماماً مع صافي الراتب الشهري (**Monthly_Inhand_Salary**)؛ حيث يبلغ **الوسيط (Median)** لفئة التصنيف الجيد (**Good**) نحو **45,000 دولار** سنوياً، تليها فئة (**Standard**) بنحو **36,000 دولار**، بينما تسجل فئة (**Poor**) أقل وسيط بنحو **31,000 دولار** سنوياً.
* ونظراً لهذا التطابق التام في التوزيع بين الدخل السنوي وصافي الراتب الشهري، سأقوم بحذف أحد العمودين في مرحلة تجهيز البيانات لتجنب تكرار المعلومات (**Redundancy**).

</div>


# --------------------------------------------------------------------------

# 1-Payment History columns 

In [ ]:
px.histogram(df_clean, x="Delay_from_due_date", title="Distribution of Delay from due date")

<div dir="rtl">

* يتضح من الرسم البياني أن النسبة الأكبر من العملاء تتأخر عن السداد لفترة تتراوح بين **5 و30 يوماً** (بذروة عند **15 يوماً**)، بينما يمتد التأخير لدى فئة أخرى ليصل إلى **67 يوماً**.
* ووفقاً لمعايير **FICO** وإدارة مخاطر الائتمان، فإن تجاوز التأخير حاجز **30 يوماً** (`30+ Days Past Due`) يُعد مؤشر خطر رئيسياً (`Red Flag`) يُسجل رسمياً في السجل الائتماني ويؤدي إلى تراجع حاد في التصنيف الائتماني (**Credit Score**).

</div>


In [ ]:
px.box(df_clean, x="Delay_from_due_date", color="Credit_Score",  title="Distribution of Delay from Due Date by Credit Score")

<div dir="rtl">

* يتضح من الرسم البياني للصندوق (**Box Plot**) وجود علاقة عكسية قوية بين زيادة أيام التأخير عن السداد (**Delay_from_due_date**) ومستوى التصنيف الائتماني (**Credit Score**)؛ حيث يتركز **50%** من عملاء فئة التصنيف المنخفض (**Poor**) بين **17 و43 يوماً** من التأخير بوسيط يبلغ **27 يوماً**.
* في المقابل، تسجل فئة (**Standard**) وسيطاً يبلغ **18 يوماً**، بينما تحافظ فئة التصنيف الجيد (**Good**) على أقل معدل تأخير بوسيط يبلغ **10 أيام** فقط (ويتركز معظمهم بين **5 و14 يوماً**).

</div>


In [ ]:
px.histogram(df_clean, x="Num_of_Delayed_Payment", title="Distribution of Num of Delayed Payment")

<div dir="rtl">

* يتضح من الرسم البياني لتوزيع عدد الدفعات المتأخرة (**Num_of_Delayed_Payment**) أن النسبة الأكبر من العملاء تتركز في النطاق ما بين **8 و20 دفعة متأخرة**، بينما يمتد التوزيع ليصل إلى **28 دفعة متأخرة** كحد أقصى.
* ووفقاً لمعايير **FICO**، يندرج هذا المتغير تحت عامل **تاريخ السداد (`Payment History`)** الذي يمثل الوزن الأكبر (**35%**) في حساب التصنيف الائتماني؛ حيث يؤدي تكرار الدفعات المتأخرة إلى تراجع حاد ومباشر في نقاط التصنيف الائتماني (**Credit Score**).

</div>


In [ ]:
px.box(df_clean, x="Num_of_Delayed_Payment", color="Credit_Score",  title="Distribution of Num of Delayed Payment by Credit Score")

<div dir="rtl">

* يتضح من الرسم البياني للصندوق (**Box Plot**) وجود علاقة عكسية قوية بين زيادة عدد الدفعات المتأخرة (**Num_of_Delayed_Payment**) ومستوى التصنيف الائتماني (**Credit Score**)؛ حيث يبلغ **الوسيط (Median)** لفئة التصنيف المنخفض (**Poor**) **17 دفعة متأخرة** (ويتركز **50%** منهم بين **12 و20 دفعة** وتصل القيم إلى **28 دفعة** كحد أقصى).
* في المقابل، ينخفض الوسيط إلى **14 دفعة متأخرة** في فئة (**Standard**)، ويسجل أدنى مستوى له في فئة التصنيف الجيد (**Good**) بوسيط يبلغ **8 دفعات متأخرة** فقط؛ مما يؤكد أن تكرار التعثر في السداد يرفع بشكل مباشر من احتمالية تصنيف العميل ضمن الفئة المنخفضة (**Poor**).

</div>


In [ ]:
px.pie(df_clean, names="Payment_of_Min_Amount", title="Distribution of Payment of Minimum Amount")

<div dir="rtl">

* يتضح من الرسم البياني الدائري (**Pie Chart**) لتوزيع سداد الحد الأدنى (**Payment_of_Min_Amount**) أن النسبة الأكبر من العملاء يكتفون بسداد الحد الأدنى فقط (`Yes`) بنسبة **59.4%**، مقابل **40.6%** من العملاء لا يكتفون بالحد الأدنى ويسددون مبالغ أكبر أو كامل المستحقات (`No`).
* ووفقاً لمعايير **FICO**، يندرج هذا المتغير ضمن عامل **تاريخ السداد (`Payment History` - 35%)**؛ حيث يشير الاكتفاء الدائم بسداد الحد الأدنى فقط إلى وجود ضغط مالي وتراكم للفوائد والديون، مما يؤثر سلباً على نقاط التصنيف الائتماني (**Credit Score**).

</div>


In [ ]:
px.histogram(df_clean, x="Payment_of_Min_Amount", color="Credit_Score",  title="Distribution of Payment of Minimum Amount by Credit Score")

<div dir="rtl">

* يتضح من الرسم البياني العلاقة القوية بين سداد الحد الأدنى (**Payment_of_Min_Amount**) والتصنيف الائتماني (**Credit Score**)؛ حيث تتركز الأغلبية العظمى من عملاء التصنيف المنخفض (**Poor**) في فئة من يكتفون بسداد الحد الأدنى فقط (`Yes`) بنحو **24 ألف سجل** مقابل **5 آلاف سجل** فقط في فئة (`No`).
* في المقابل، يتركز معظم أصحاب التصنيف الجيد (**Good**) في فئة عدم الاكتفاء بالحد الأدنى (`No`) بنحو **15 ألف سجل** ويندر وجودهم في فئة (`Yes`) بنحو **2,500 سجل** فقط، بينما تتقارب أعداد فئة (**Standard**) بين الفئتين؛ مما يؤكد أن سداد أكثر من الحد الأدنى يرفع من موثوقية العميل وتصنيفه الائتماني.

</div>


In [ ]:
px.pie(df_clean, names="Payment_Behaviour",   title="Distribution of payment Behaviour")


<div dir="rtl">

* يتضح من الرسم البياني الدائري (**Pie Chart**) لتوزيع سلوك الإنفاق والسداد (**Payment_Behaviour**) أن النسبة الأكبر من العملاء تندرج تحت فئة الإنفاق المنخفض والمدفوعات صغيرة القيمة (`Low_spent_Small_value_payments`) بنسبة **27.8%** (أكثر من ربع العملاء بقليل)، تليها فئة الإنفاق المرتفع والمدفوعات متوسطة القيمة (`High_spent_Medium_value_payments`) بنسبة **19.4%**، بينما تمثل فئة الإنفاق المنخفض والمدفوعات كبيرة القيمة (`Low_spent_Large_value_payments`) النسبة الأقل بواقع **10.9%**.
* ووفقاً لمعايير **FICO**، يرتبط هذا المتغير السلوكي بعاملي **تاريخ السداد (`Payment History` - 35%)** و**حجم الديون المستغلة (`Amounts Owed` - 30%)**؛ حيث يعكس التوازن بين حجم الإنفاق وقيمة السداد قدرة العميل على التحكم في التزاماته وتجنب تراكم المديونية.

</div>


In [ ]:
px.histogram(df_clean,x="Payment_Behaviour",color="Credit_Score",barmode="group", title="Payment Behaviour by Credit Score")

<div dir="rtl">

* يتضح من الرسم البياني أن أسلوب الإنفاق المنخفض والمدفوعات صغيرة القيمة (`Low_spent_Small_value_payments`) هو الأكثر انتشاراً بين العملاء عبر جميع فئات التصنيف الائتماني، يليه أسلوب الإنفاق المرتفع والمدفوعات متوسطة القيمة (`High_spent_Medium_value_payments`).
* كما يُلاحظ تقارب نسب توزع فئات التصنيف (**Standard** ثم **Poor** ثم **Good**) داخل كل أسلوب سلوكي بشكل متشابه؛ مما يشير إلى أن سلوك الإنفاق بمفرده لا يمثل فاصلاً حاسماً بين الفئات دون دمجه مع مؤشرات الالتزام بالسداد وحجم الديون.

</div>


========================================================================================================================================

<div dir="rtl">

### ملخص تحليل تاريخ السداد (Payment History Summary):

* يتضح من الرسومات البيانية أن التصنيف الائتماني (**Credit Score**) يرتبط بشكل طردي ومباشر بمدى التزام العميل بسداد التزاماته في مواعيدها؛ فكلما كانت مؤشرات الدفع والانتظام إيجابية (انخفاض أيام التأخير، قلة الدفعات المتأخرة، وعدم الاكتفاء بسداد الحد الأدنى)، ارتفع التصنيف الائتماني للعميل نحو فئة (**Good**)، والعكس صحيح.

</div>


=========================================================================================================================================

# 2- Amounts Owed Conclusion columns

In [ ]:
px.histogram(df_clean, x="Credit_Utilization_Ratio", title="Distribution of Credit Utilization Ratio (%)")

<div dir="rtl">

* يتضح من الرسم البياني لتوزيع نسبة استغلال الائتمان (**Credit_Utilization_Ratio**) أن التوزيع متماثل تقريباً ويقترب من التوزيع الطبيعي المعتدل (**Normal-like Distribution**)؛ حيث يتركز معظم العملاء في النطاق ما بين **25% و40%** بمتوسط يتراوح حول **30% إلى 35%**.
* ووفقاً لمعايير **FICO**، يندرج هذا المتغير تحت عامل **حجم الديون المستغلة (`Amounts Owed` - 30%)**؛ حيث تنصح القاعدة المالية بعدم تجاوز نسبة الاستغلال حاجز **30%**، مما يشير إلى أن غالبية العملاء في البيانات يستهلكون حدودهم الائتمانية بمعدلات تتجاوز النطاق الآمن.

</div>


In [ ]:
px.box(df_clean, x="Credit_Utilization_Ratio", color="Credit_Score",  title="Distribution of Credit Utilization Ratio by Credit Score")

<div dir="rtl">

* يتضح من الرسم البياني للصندوق (**Box Plot**) تطابق توزيع نسبة استغلال الائتمان (**Credit_Utilization_Ratio**) بشكل كبير عبر جميع فئات التصنيف الائتماني (**Good**, **Standard**, **Poor**)؛ حيث يتقارب **الوسيط (Median)** بين الفئات الثلاثة حول **32% إلى 33%** ويقع معظم العملاء في نفس النطاق ما بين **28% و37%**.
* ويشير هذا التقارب الشديد إلى أن نسبة استغلال الائتمان في هذه البيانات لا تمثل بمفردها عاملاً فاصلاً قوياً بين فئات التصنيف الائتماني دون دمجها مع باقي مؤشرات المديونية والسداد.

</div>


In [ ]:
px.histogram(df_clean, x="Outstanding_Debt", title="Distribution of Outstanding Debt")

<div dir="rtl">

#### من خلال الرسم البياني، يتضح أن النسبة الأكبر من العملاء لا تتجاوز ديونهم المستحقة (Outstanding Debt) قيمة 1,600 دولار.
#### كما يُلاحظ وجود فئة من العملاء تصل قيمة ديونهم المستحقة إلى نحو 5,000 دولار (مما يشير إلى وجود التواء يميني Right Skewness في التوزيع).

</div>


In [ ]:
px.box(df_clean, x="Outstanding_Debt", color="Credit_Score",  title="Distribution of Outstanding Debt by Credit Score")

<div dir="rtl">

من خلال تحليل الرسم البياني للصندوق (Box Plot)، يتضح أن قيمة الوسيط (Median) للديون المستحقة (Outstanding Debt) لدى الفئة ذات التصنيف المنخفض (Poor) تفوق نظيرتها في فئتي (Standard) و (Good) بشكل ملحوظ؛ حيث تصل قيمة الديون في فئة (Poor) إلى نحو 2,000 دولار، مقارنة بنحو 1,000 دولار لفئة (Standard)، وحوالي 740 دولاراً لفئة (Good). وتأسيساً على المفاهيم المالية المعتمدة في التقييم الائتماني، فإن ارتفاع حجم المديونية يعكس تعرض العميل لضغوط مالية متزايدة، مما يميل إلى التأثير سلباً على نقاط التصنيف الائتماني (Credit Score) ويؤدي إلى فقدان النقاط.

</div>


In [ ]:
px.histogram(df_clean, x="Total_EMI_per_month", title="Distribution of Total EMI per Month")

<div dir="rtl">

* نلاحظ من الرسم البياني وجود التواء يميني حاد (**Right Skewness**) في التوزيع؛ حيث يتركز معظم العملاء في نطاق الأقساط الشهرية المنخفضة (أقل من **200 دولار**)، مع وجود نسبة كبيرة عند القيمة صفر (عملاء ليس لديهم أقساط شهرية)، بينما تمتد القيم المرتفعة بشكل متناقص حتى تصل إلى **1,780 دولار**.

</div>


In [ ]:
px.box(df_clean, x="Total_EMI_per_month", color="Credit_Score",  title="Distribution of Total EMI per Month by Credit Score")

<div dir="rtl">

### تحليل توزيع الأقساط الشهرية حسب التصنيف الائتماني (Total EMI per Month by Credit Score):

* **الوسيط (Median):** يتضح من الرسم البياني للصندوق (**Box Plot**) وجود علاقة طردية بين ارتفاع الأقساط الشهرية وانخفاض التصنيف الائتماني؛ حيث سجلت فئة التصنيف المنخفض (**Poor**) أعلى قيمة وسيط للأقساط الشهرية بنحو **74.8 دولار** (ويصل الربع الثالث **Q3** إلى **157.7 دولار**)، تليها فئة (**Standard**)، بينما سجلت فئة التصنيف الجيد (**Good**) أقل التزامات شهرية.
* **التشتت والالتواء (Spread & Skewness):** تُظهر البيانات وجود التواء يميني (**Right Skewness**) واضح في جميع الفئات، إلا أن فئة (**Poor**) تتميز بأكبر نطاق تشتت، مع امتداد القيم المرتفعة للأقساط لتصل إلى نحو **1,634 دولاراً**.
* **الاستنتاج المالي والنمذجة (Financial & Modeling Insight):** يعكس ارتفاع قيمة الـ **Total EMI per Month** زيادة الضغط على السيولة الشهرية للعميل وارتفاع مخاطر التعثر، مما يجعل هذا المتغير (وخاصة عند دمجه مع الدخل الشهري) ذا قدرة تمييزية جيدة لفصل الفئات في نماذج التنبؤ.

</div>



In [ ]:
px.histogram(df_clean, x="Monthly_Balance", title="Distribution of monthly Balance")

<div dir="rtl">

* يُلاحظ من الرسم البياني أن النسبة الأكبر من العملاء يتركز الرصيد الشهري المتبقي (**Monthly_Balance**) لديهم حول قيمة **280 دولاراً**، كما يظهر التوزيع وجود التواء يميني (**Right Skewness**)؛ حيث تمتد القيم المتبقية بشكل متناقص لتصل إلى نحو **1,600 دولار**.

</div>


In [ ]:
px.box(df_clean, x="Monthly_Balance", color="Credit_Score",  title="Distribution of Monthly Balance by Credit Score")

<div dir="rtl">

* من خلال الرسم البياني للصندوق (**Box Plot**)، يتضح وجود تباين في توزيع الرصيد الشهري المتبقي (**Monthly_Balance**) بين الفئات المختلفة؛ حيث يبلغ **الوسيط (Median)** لفئة التصنيف الجيد (**Good**) نحو **400 دولار** شهرياً، تليها فئة (**Standard**) بنحو **350 دولاراً**، بينما تسجل فئة التصنيف المنخفض (**Poor**) أقل وسيط بنحو **300 دولار** شهرياً.
* ووفقاً لمبادئ **إدارة مخاطر الائتمان (Credit Risk Management)**، فإن ارتفاع صافي السيولة المتبقية في نهاية الشهر يعكس تمتع العميل بمرونة مالية (**Financial Liquidity**) وقدرة أعلى على الوفاء بالالتزامات الطارئة دون ضغوط، مما يقلل من احتمالية التعثر (**Default Risk**) ويعزز فرص حصوله على تصنيف ائتماني مرتفع.

</div>


=========================================================================================================

<div dir="rtl">

### الاستنتاج العام لقسم حجم الديون المستغلة (Amounts Owed Conclusion):

* **الأثر المباشر للمديونية:** تُظهر نتائج التحليل الاستكشافي (**EDA**) وجود علاقة عكسية واضحة بين حجم الالتزامات المالية ومستوى التصنيف الائتماني (**Credit Score**)؛ حيث ترتفع قيم إجمالي الديون المستحقة (**Outstanding Debt**) والأقساط الشهرية الثابتة (**Total EMI per Month**) بشكل ملحوظ لدى عملاء فئة التصنيف المنخفض (**Poor**) مقارنة بفئتي (**Standard**) و(**Good**).
* **سلوك استغلال الائتمان والسيولة:** يتضح أن غالبية العملاء يتجاوزون النسبة الآمنة لاستغلال الائتمان البالغة **30%** (**Credit Utilization Ratio**)، كما ينعكس ارتفاع الأعباء المالية مباشرة على انخفاض صافي الرصيد الشهري المتبقي (**Monthly Balance**) لدى الفئات الأقل تصنيفاً.
* **الأهمية للنمذجة (Modeling Relevance):** تُثبت هذه المؤشرات مجتمعة أن متغيرات حجم المديونية والسيولة تمثل عوامل حاسمة ذات قدرة تمييزية عالية (**High Predictive Power**) للفصل بين مستويات المخاطر الائتمانية، مما يجعلها ركيزة أساسية في بناء نماذج التنبؤ.

</div>


=====================================================================================================================

# 3- Credit History Length Column

In [ ]:
df_clean['Credit_History_Age'] = (df_clean['Credit_History_Age'] / 12).astype(int)

<div dir="rtl">

* تم تحويل وحدة قياس عمر التاريخ الائتماني (**Credit_History_Age**) من الشهور إلى السنوات (بالقسمة على 12) لتسهيل تفسير القيم وقراءة المحاور في الرسوم البيانية بشكل أوضح وأكثر واقعية، دون التأثير على التوزيع الإحصائي الأصلي للبيانات.

</div>


In [ ]:
px.histogram(df_clean, x='Credit_History_Age', title="Distribution of Credit_History_Age")

<div dir="rtl">

* يتضح من الرسم البياني أن النسبة الأكبر من العملاء يتجاوز عمر تاريخهم الائتماني (**Credit_History_Age**) **15 عاماً**؛ حيث تتركز أعلى كثافة للبيانات في الفترة ما بين **15 و20 عاماً**، ويمتد التوزيع ليصل إلى نحو **33 عاماً**.

</div>


In [ ]:
px.box(df_clean, x="Credit_History_Age", color="Credit_Score",  title="Distribution of Credit History Age by Credit Score")

<div dir="rtl">

* يتضح من الرسم البياني للصندوق (**Box Plot**) وجود علاقة طردية واضحة بين طول عمر التاريخ الائتماني (**Credit_History_Age**) وارتفاع التصنيف الائتماني (**Credit Score**)؛ حيث يبلغ **الوسيط (Median)** لفئة (**Good**) نحو **24 عاماً**، مقارنة بنحو **18 عاماً** لفئة (**Standard**)، ونحو **13 عاماً** لفئة (**Poor**).
* كما يُلاحظ وجود قيم متطرفة منخفضة (**Outliers**) في فئة (**Good**) لأعمار ائتمانية تقل عن **6 سنوات**؛ مما يشير إلى إمكانية حصول العميل ذي التاريخ الائتماني القصير على تصنيف مرتفع في حال التزامه القوي بباقي العوامل (مثل انتظام السداد وانخفاض المديونية).

</div>


=======================================================================================================================

# 4- Credit Mix Columns

In [ ]:
px.pie(df_clean, names='Credit_Mix', title="Distribution of Credit_Mix")

<div dir="rtl">

* يتضح من الرسم البياني الدائري (**Pie Chart**) لتوزيع تنوع الائتمان (**Credit_Mix**) أن النسبة الأكبر من العملاء تندرج تحت التصنيف المتوسط (**Standard**) بنسبة **45.8%** (مما يعكس تنوعاً متوسطاً في الحسابات الائتمانية)، تليها فئة التنوع الجيد (**Good**) بنسبة **30.4%**، بينما تمثل فئة التنوع الضعيف (**Bad**) النسبة الأقل بواقع **23.8%**.

</div>


In [ ]:
px.histogram(df_clean, x='Credit_Mix', color='Credit_Score', barmode='group', histnorm='percent', title='Credit Mix vs Credit Score (%)')



<div dir="rtl">

* يتضح من الرسم البياني وجود علاقة طردية قوية بين جودة تنوع الائتمان (**Credit_Mix**) وارتفاع التصنيف الائتماني (**Credit Score**)؛ حيث تتركز الأغلبية العظمى من أصحاب التصنيف الجيد (**Good**) بنسبة **83.3%** ضمن فئة التنوع الجيد (**Good Credit Mix**)، بينما تنخفض نسبتهم بشكل حاد إلى **2.0%** فقط في فئة التنوع الضعيف (**Bad**).
* في المقابل، تتركز النسبة الأكبر من أصحاب التصنيف المتوسط (**Standard**) بنسبة **62.7%** في فئة التنوع المتوسط، بينما تتركز النسبة الأعلى من أصحاب التصنيف المنخفض (**Poor**) بنسبة **49.3%** ضمن فئة التنوع الضعيف (**Bad**).

</div>


In [ ]:
px.histogram(df_clean, x="Num_Credit_Card", title="Distribution of Num Credit Card")

<div dir="rtl">

* يتضح من الرسم البياني أن النسبة الأكبر من العملاء يمتلكون ما بين **3 إلى 7 بطاقات ائتمانية** (`Num_Credit_Card`)؛ حيث تبلغ الذروة عند **5 بطاقات**، تليها فئة العملاء الذين يمتلكون **6 و7 بطاقات**، بينما يمتد التوزيع ليصل إلى **10 بطاقات**.
* ووفقاً للمبادئ المالية المعتمدة في التقييم الائتماني (**Credit Risk Principles**)، فإن امتلاك أكثر من **5 بطاقات ائتمانية** يُعد مؤشراً سلبياً يرفع من احتمالية التوسع المفرط في الاقتراض وزيادة الطلبات الائتمانية الجديدة، مما ينعكس سلباً على نقاط التصنيف الائتماني (**Credit Score**) للعميل.

</div>


In [ ]:
px.box(df_clean, x="Num_Credit_Card", color="Credit_Score",  title="Distribution of Num Credit Card by Credit Score")

<div dir="rtl">

* يتضح من الرسم البياني للصندوق (**Box Plot**) وجود علاقة عكسية بين زيادة عدد بطاقات الائتمان (**Num_Credit_Card**) ومستوى التصنيف الائتماني (**Credit Score**)؛ حيث يبلغ **الوسيط (Median)** لفئة التصنيف الجيد (**Good**) **4 بطاقات** (ويتركز معظمهم بين **3 و5 بطاقات**)، بينما يرتفع الوسيط إلى **5 بطاقات** في فئة (**Standard**)، ويصل إلى **7 بطاقات** في فئة التصنيف المنخفض (**Poor**).
* ويؤكد هذا التوزيع أن الحفاظ على عدد معتدل من البطاقات (أقل من **5 بطاقات**) يعزز من الاستقرار والتقييم المالي للعميل، في حين أن التوسع في امتلاك البطاقات يرتبط بزيادة المخاطر الائتمانية وتراجع التصنيف.

</div>


In [ ]:
px.histogram(df_clean, x="Num_Bank_Accounts", title="Distribution of Num Bank Accounts")   

<div dir="rtl">

* يتضح من الرسم البياني لتوزيع عدد الحسابات البنكية (**Num_Bank_Accounts**) أن النسبة الأكبر من العملاء يمتلكون ما بين **3 إلى 8 حسابات بنكية** (حيث تبلغ الذروة عند **6 إلى 8 حسابات**)، بينما يمتد التوزيع الإجمالي للعملاء من **0** وحتى **10 حسابات**.
* ووفقاً لمبادئ **إدارة مخاطر الائتمان (`Credit Risk Management`)**، فإن التوسع في فتح عدد كبير من الحسابات البنكية قد يعكس تشتت السيولة وصعوبة إدارة الالتزامات المالية، مما يرتبط غالباً بارتفاع احتمالية المخاطر الائتمانية مقارنة بالعملاء الذين يحافظون على عدد معتدل ومنظم من الحسابات.

</div>


In [ ]:
px.box(df_clean , x="Num_Bank_Accounts", color="Credit_Score",  title="Distribution of Num Bank Accounts by Credit Score")

<div dir="rtl">

* يتضح من الرسم البياني للصندوق (**Box Plot**) وجود علاقة عكسية واضحة بين زيادة عدد الحسابات البنكية (**Num_Bank_Accounts**) ومستوى التصنيف الائتماني (**Credit Score**)؛ حيث يبلغ **الوسيط (Median)** لفئة التصنيف الجيد (**Good**) **3 حسابات بنكية** (ويتركز معظمهم بين **2 و5 حسابات**)، بينما يرتفع الوسيط إلى **5 حسابات** في فئة (**Standard**)، ويصل إلى **7 حسابات** في فئة التصنيف المنخفض (**Poor**).
* ووفقاً لمبادئ **إدارة مخاطر الائتمان (`Credit Risk Management`)**، يؤكد هذا التوزيع أن الحفاظ على عدد معتدل ومنظم من الحسابات البنكية يعزز الاستقرار المالي للعميل، بينما يرتبط التوسع المفرط في فتح الحسابات بتشتت السيولة وزيادة احتمالية التعثر وتراجع التصنيف.

</div>


In [ ]:
px.histogram(df_clean, x="Num_of_Loan", title="Distribution of Num of Loan")

<div dir="rtl">

* يتضح من الرسم البياني لتوزيع عدد القروض (**Num_of_Loan**) أن النسبة الأكبر من العملاء يمتلكون ما بين **0 إلى 4 قروض** (حيث تتركز الذروة عند **2 إلى 4 قروض**)، بينما يمتد الذيل اليميني للتوزيع (**Right Skewness**) بشكل متناقص ليصل إلى **9 قروض**.
* ووفقاً لمبادئ **إدارة مخاطر الائتمان (`Credit Risk Management`)**، فإن الحفاظ على عدد محدود ومنتظم من القروض يعكس قدرة العميل على إدارة التزاماته بأقساط ثابتة دون ضغوط، في حين أن التوسع المفرط في تعدد القروض يرفع من عبء المديونية الشهرية ويزيد من احتمالية التعثر، مما ينعكس سلباً على التصنيف الائتماني (**Credit Score**).

</div>


In [ ]:
px.box(df_clean, x="Num_of_Loan", color="Credit_Score",  title="Distribution of Num of Loan by Credit Score")

<div dir="rtl">

* يتضح من الرسم البياني للصندوق (**Box Plot**) وجود علاقة عكسية واضحة بين زيادة عدد القروض (**Num_of_Loan**) ومستوى التصنيف الائتماني (**Credit Score**)؛ حيث يبلغ **الوسيط (Median)** لفئة التصنيف الجيد (**Good**) **قرضين (2)** فقط (ويتركز معظمهم بين **1 و3 قروض**)، بينما يرتفع الوسيط إلى **3 قروض** في فئة (**Standard**)، ويصل إلى **5 قروض** في فئة التصنيف المنخفض (**Poor**).
* كما يُلاحظ وجود قيم متطرفة مرتفعة (**Outliers**) في فئة (**Good**) لعملاء يمتلكون من **7 إلى 9 قروض**؛ مما يشير إلى إمكانية الحفاظ على تصنيف ائتماني مرتفع رغم تعدد القروض في حال الالتزام التام بالسداد في المواعيد المحددة وعدم تعثر الدفعات.

</div>


==============================================================================================================================

<div dir="rtl">

### ملخص تحليل أنواع الائتمان (Credit Mix Summary):

* **جودة تنوع المحفظة (`Credit_Mix`):** أظهر التحليل وجود علاقة طردية قوية بين جودة تنوع الائتمان والتصنيف الائتماني؛ حيث تركزت نسبة **83.3%** من أصحاب التصنيف الجيد (**Good**) ضمن فئة التنوع الجيد، في مقابل تركز **49.3%** من أصحاب التصنيف المنخفض (**Poor**) في فئة التنوع الضعيف (**Bad**).
* **تعدد الحسابات والبطاقات (`Num_Credit_Card` & `Num_Bank_Accounts`):** اتضح وجود علاقة عكسية واضحة بين التوسع المفرط في امتلاك البطاقات والحسابات البنكية ومستوى التصنيف الائتماني؛ حيث يحافظ عملاء الفئة الجيدة (**Good**) على عدد معتدل بوسيط (**4 بطاقات** و **3 حسابات بنكية**)، بينما يرتفع الوسيط لدى الفئة المنخفضة (**Poor**) ليصل إلى (**7 بطاقات** و **7 حسابات بنكية**)، مما يعكس تشتت السيولة وزيادة احتمالية المخاطر.
* **عدد القروض (`Num_of_Loan`):** أظهرت النتائج أن زيادة عدد القروض القائمة تؤثر سلباً على التصنيف الائتماني بوسيط **قرضين** لفئة (**Good**) مقابل **5 قروض** لفئة (**Poor**)، مع ملاحظة استثناءات تؤكد أن الالتزام بالسداد يظل العامل الحاسم حتى مع تعدد القروض.

</div>


======================================================================================================================

# 5- New Credit Columns

In [ ]:
px.histogram(df_clean, x="Num_Credit_Inquiries", title="Distribution of num of Credit Inquiries")

<div dir="rtl">

* يتضح من الرسم البياني لتوزيع عدد الاستعلامات الائتمانية (**Num_Credit_Inquiries**) أن النسبة الأكبر من العملاء يقع عدد استعلاماتهم في النطاق ما بين **0 و6 استعلامات** (حيث تبلغ الذروة عند **4 استعلامات**)، بينما يمتد الذيل اليميني للتوزيع (**Right Skewness**) بشكل متناقص ليصل إلى **17 استعلاماً**.
* ووفقاً لمبادئ **إدارة مخاطر الائتمان (`Credit Risk Management`)** ونموذج **FICO**، فإن الحفاظ على عدد محدود من الاستعلامات يعكس استقراراً مالياً، في حين أن تكرار الاستعلامات الصارمة (`Hard Inquiries`) يشير إلى طلب متزايد على السيولة وارتفاع احتمالية التعثر، مما ينعكس سلباً على نقاط التصنيف الائتماني (**Credit Score**).

</div>



In [ ]:
px.box(df_clean, x="Num_Credit_Inquiries", color="Credit_Score",  title="Distribution of Num Credit Inquiries by Credit Score")

<div dir="rtl">

* يتضح من الرسم البياني للصندوق (**Box Plot**) وجود علاقة عكسية واضحة بين زيادة عدد الاستعلامات الائتمانية (**Num_Credit_Inquiries**) ومستوى التصنيف الائتماني (**Credit Score**)؛ حيث يبلغ **الوسيط (Median)** لفئة التصنيف الجيد (**Good**) **3 استعلامات** فقط (ويتركز معظمهم بين **1 و4 استعلامات**)، بينما يرتفع الوسيط إلى **5 استعلامات** في فئة (**Standard**)، ويصل إلى **8 استعلامات** في فئة التصنيف المنخفض (**Poor**).
* كما يُلاحظ وجود قيم متطرفة مرتفعة (**Outliers**) في فئة (**Good**) لعملاء لديهم من **9 إلى 16 استعلاماً**؛ مما يشير إلى إمكانية الحفاظ على تصنيف ائتماني مرتفع رغم كثرة الاستعلامات في حال قوة باقي العوامل الأساسية (مثل انتظام السداد وانخفاض المديونية).

</div>


In [ ]:
px.histogram(df_clean, x='Changed_Credit_Limit', title="Distribution of Changed Credit Limit % ")

<div dir="rtl">

* يتضح من الرسم البياني لتوزيع نسبة التغيير في الحد الائتماني (**Changed_Credit_Limit**) أن النسبة الأكبر من العملاء تتركز في نطاق الزيادة الإيجابية ما بين **0% و12%** تقريباً (حيث تبلغ الذروة حول **7% إلى 11%**)، بينما يمتد الذيل اليميني للتوزيع (**Right Skewness**) ليصل إلى نحو **36%**، مع وجود نسبة محدودة جداً من العملاء شهدت تخفيضاً في الحد الائتماني بقيم سالبة تصل إلى **-6%**.
* ووفقاً لمبادئ **إدارة مخاطر الائتمان (`Credit Risk Management`)**، فإن الزيادة المعتدلة في الحد الائتماني تعكس ثقة المؤسسات المالية في ملاءة العميل وقدرته على السداد، مما يساهم في خفض **نسبة استغلال الائتمان (`Credit Utilization Ratio`)**، في حين يشير تخفيض الحد الائتماني (القيم السالبة) إلى إجراء وقائي تتخذه البنوك للحد من مخاطر التعثر عند رصد مؤشرات سلبية في سلوك العميل.

</div>


In [ ]:
px.box(df_clean, x="Changed_Credit_Limit", color="Credit_Score",  title="Distribution of Changed Credit Limit by Credit Score")

<div dir="rtl">

* يتضح من الرسم البياني للصندوق (**Box Plot**) تقارب توزيع نسبة التغيير في الحد الائتماني (**Changed_Credit_Limit**) بين فئتي (**Standard**) و(**Poor**) بوسيط يبلغ نحو **10.3%** و **9.7%** على التوالي ونطاق تشتت أوسع، بينما تتميز فئة التصنيف الجيد (**Good**) باستقرار أكبر ونطاق أضيق بوسيط يبلغ نحو **6.7%** (ويتركز معظمهم بين **3.5% و10%**).
* ويعكس هذا النمط أن عملاء الفئة الجيدة يتمتعون باستقرار مالي ولا يميلون لطلب تعديلات حادة في حدودهم الائتمانية، في حين يشير التشتت الأوسع لدى الفئات الأقل تصنيفاً إلى تذبذب سلوكهم الائتماني ومحاولاتهم المتكررة لزيادة سقف الائتمان، مع ملاحظة ضعف القدرة التمييزية لهذا المتغير بمفرده في الفصل بين فئتي (**Standard**) و(**Poor**).

</div>


===================================================================================================================

<div dir="rtl">

### ملخص تحليل طلبات الائتمان الجديدة (New Credit Summary):

* **عدد الاستعلامات الائتمانية (`Num_Credit_Inquiries`):** أظهر التحليل وجود علاقة عكسية واضحة بين كثرة الاستعلامات الائتمانية ومستوى التصنيف الائتماني (**Credit Score**)؛ حيث يرتفع **الوسيط (Median)** من **3 استعلامات** فقط لدى فئة التصنيف الجيد (**Good**) ليصل إلى **5 استعلامات** في فئة (**Standard**)، و**8 استعلامات** في فئة التصنيف المنخفض (**Poor**)، مما يؤكد أن تكرار طلب الائتمان يُعد مؤشراً قوياً على الضغط المالي وزيادة المخاطر.
* **التغيير في الحد الائتماني (`Changed_Credit_Limit`):** اتضح أن عملاء الفئة الجيدة (**Good**) يتمتعون باستقرار أكبر في حدودهم الائتمانية بوسيط يبلغ نحو **6.7%** ونطاق تشتت ضيق، مقارنة بتذبذب أوسع ووسيط أعلى لدى فئتي (**Standard**) و(**Poor**) بنحو **10.3%** و**9.7%** على التوالي؛ مما يعكس ميل الفئات الأقل تصنيفاً لطلب تعديلات متكررة في سقف الائتمان، مع محدودية القدرة التمييزية لهذا المتغير بمفرده في الفصل بين فئتي (**Standard**) و(**Poor**).

</div>


==========================================================================================================================

# Heatmap

In [ ]:
px.imshow(numaric_df.corr(), text_auto=True, aspect="auto", color_continuous_scale="RdBu_r", title="Correlation Matrix")

<div dir="rtl">

### تحليل مصفوفة الارتباط (Correlation Matrix ):

* **التعدد الخطي المرتفع (`High Multicollinearity`):** يُظهر الرسم البياني ارتباطاً طردياً شبه تام بين الدخل السنوي (`Annual_Income`) وصافي الراتب الشهري (`Monthly_Inhand_Salary`) بمعامل ارتباط يصل إلى **0.998**؛ مما يستدعي الاحتفاظ بأحدهما فقط في مرحلة إعداد البيانات للنماذج (`Feature Selection`) لتجنب تكرار المعلومات.
* **ارتباط مؤشرات المخاطر الائتمانية:** يظهر ارتباط طردي متوسط إلى قوي (يتراوح بين **0.50 و 0.65**) بين متغيرات الخطر المتمثلة في معدل الفائدة (`Interest_Rate`)، إجمالي الديون المستحقة (`Outstanding_Debt`)، تأخير السداد (`Delay_from_due_date`)، وعدد الاستعلامات الائتمانية (`Num_Credit_Inquiries`).
* **العلاقة العكسية مع الاستقرار الائتماني:** يرتبط طول عمر التاريخ الائتماني (`Credit_History_Age`) بعلاقة عكسية واضحة مع مؤشرات المخاطر (خاصة `Outstanding_Debt` بمعامل **-0.63** و `Num_Credit_Inquiries` بمعامل **-0.61**)؛ مما يؤكد أن أصحاب التاريخ الائتماني الطويل يتمتعون بمديونية أقل واستقرار مالي أعلى.

</div>


========================================================================================================================

<div dir="rtl">

### ملخص مرحلة التحليل الاستكشافي (EDA Summary):

بناءً على تحليل البيانات وفقاً لعوامل **FICO الخمسة**، توصلنا إلى النتائج التالية:

* **1. تاريخ السداد (`Payment History`):** اتضح أن التأخير في السداد وعدد الدفعات المتأخرة هما العاملان الأكثر تأثيراً في تراجع التصنيف الائتماني نحو فئة (**Poor**)، بينما يتميز أصحاب فئة (**Good**) بالالتزام المنتظم في المواعيد.
* **2. حجم الديون المستغلة (`Amounts Owed`):** تبين وجود علاقة عكسية واضحة؛ حيث ترتفع الديون المستحقة (**Outstanding_Debt**) والأقساط الشهرية (**Total_EMI_per_month**) بشكل ملحوظ لدى فئة (**Poor**) وتنخفض السيولة المتبقية (**Monthly_Balance**)، على عكس فئة (**Good**) التي تحافظ على ديون أقل وسيولة أعلى.
* **3. طول تاريخ الائتمان (`Credit History Length`):** أظهر التحليل أن زيادة عمر التاريخ الائتماني (**Credit_History_Age**) ترفع من موثوقية العميل وتصنيفه الائتماني (بوسيط **24 عاماً** لفئة **Good** مقابل **13 عاماً** لفئة **Poor**).
* **4. أنواع الائتمان (`Credit Mix`):** تبين أن التنوع المتوازن في المحفظة الائتمانية مع الحفاظ على عدد معتدل من البطاقات الائتمانية والحسابات البنكية والقروض يميز عملاء فئة (**Good**)، في حين يرتبط التوسع المفرط في الحسابات والبطاقات بفئة (**Poor**).
* **5. طلبات الائتمان الجديدة (`New Credit`):** اتضح أن كثرة الاستعلامات الائتمانية (**Num_Credit_Inquiries**) والتذبذب الكبير في تعديل الحدود الائتمانية (**Changed_Credit_Limit**) يشيران إلى ضغط مالي ويرتبطان بانخفاض التصنيف الائتماني.

</div>


==================================================================================================================================

# Feature Engineering

========================================================================================================================================

<div dir="rtl">

### 1. تاريخ السداد (`Payment History` - 35%):

* **القاعدة المالية المعتمدة (`The 30-Day Delinquency Rule`):** وفقاً لمعايير **FICO**، يُعد تاريخ السداد العامل الأكبر تأثيراً في التقييم الائتماني (**35%**). ولا تُسجل مخالفة التأخير رسمياً في التقرير الائتماني إلا بعد تجاوز حاجز **30 يوماً** على تاريخ الاستحقاق (`30+ Days Past Due`)، حيث تتضاعف خطورة التعثر مع الانتقال إلى شرائح التأخير الأعلى (30 يوماً، 60 يوماً فأكثر) وتكرار مرات التأخير، بالإضافة إلى الاكتفاء بسداد الحد الأدنى فقط (`Payment of Minimum Amount`) الذي يعكس تعثر العميل في سداد أصل الدين.
* **قرار الهندسة (`Feature Creation & Selection`):** تم دمج متغيرات التأخير وسداد الحد الأدنى في مؤشر رقمي واحد (**`Payment_History_Risk`**) يقيس إجمالي مخاطر السداد، مع حذف المتغيرات الفرعية لتقليل الأبعاد وتجنب تكرار المعلومات.

</div>


In [ ]:
df_FE = df_clean.copy()

delay_tier = (df_FE['Delay_from_due_date'] // 30).clip(lower=0)
min_payment_penalty = (df_FE['Payment_of_Min_Amount'] == 'Yes').astype(int) * 5
df_FE['Payment_History_Risk'] = (df_FE['Num_of_Delayed_Payment'] * (delay_tier + 1)) + min_payment_penalty
df_FE.drop(columns=[ 'Payment_of_Min_Amount', 'Payment_Behaviour'], inplace=True)


<div dir="rtl">

### تفسير الكود:

* **1-** أخذنا نسخة من البيانات النظيفة (`df_clean`) لكي تكون مرجعاً لنا في حالة حدوث أي خطأ.
* **2-** أنشأنا متغيراً لوضع ناتج القسمة الصحيحة لعمود أيام التأخير (`Delay_from_due_date`) على **30 يوماً** (مع ضبط الحد الأدنى عند الصفر).
* **3-** أنشأنا عقوبة (`Penalty`) بحيث يحصل العميل على **5 نقاط خطر** إذا كان يدفع الحد الأدنى فقط، و **0 نقاط** إذا كان يدفع أكثر من الحد الأدنى.
* **4-** عرفنا العمود الجديد (`Payment_History_Risk`) بضرب **عدد الدفعات المتأخرة** في `(شريحة التأخير على 30 + 1)`، ثم جمع الناتج على العقوبة (`Penalty`) التي أنشأناها.
* **5-** حذفنا عمودين ليس لهما قيمة إضافية للنموذج بعد إنشاء عمود `Payment_History_Risk`.

</div>


In [ ]:
px.box(df_FE, x="Payment_History_Risk", color="Credit_Score",  title="Distribution of Payment History Risk by Credit Score")

<div dir="rtl">

* يتضح من الرسم البياني للصندوق (**Box Plot**) نجاح الميزة المستحدثة (**Payment_History_Risk**) في الفصل بوضوح بين فئات التصنيف الائتماني وفقاً لقاعدة الـ **30 يوماً** في نموذج **FICO**؛ حيث تسجل فئة التصنيف الجيد (**Good**) أقل مستوى مخاطر بوسيط يبلغ **8 نقاط** فقط (بين **4 و13 نقطة**)، يرتفع إلى **18 نقطة** في فئة (**Standard**)، ويقفز إلى **25 نقطة** في فئة التصنيف المنخفض (**Poor**) مع امتداد التشتت ليصل إلى **77 نقطة** نتيجة تضاعف أوزان التأخير طويل الأمد.

</div>


=========================================================================================================================================

<div dir="rtl">

### 2.  حجم الديون المستغلة (`Amounts Owed` - 30%):

* **القاعدة المالية المعتمدة (`Debt-to-Income & Utilization Rules`):** وفقاً لمعايير **FICO**، يُمثل حجم الديون المستغلة ثاني أكبر عامل مؤثر في التقييم الائتماني (**30%**). ولا يُقاس خطر المديونية بحجم الدين المطلق فقط، بل بنسبة إجمالي الديون والالتزامات الشهرية إلى الدخل (`Debt-to-Income Ratio`)، بالإضافة إلى نسبة استهلاك الحدود الائتمانية المتاحة (`Credit Utilization Ratio`)؛ حيث تُوصي القواعد المالية بألا يتجاوز استهلاك الائتمان حاجز **30%**، وألا تلتهم الأقساط الشهرية أكثر من **35% إلى 40%** من صافي الدخل الشهري، لأن تجاوز هذه الحدود يقلص السيولة المتبقية ويرفع احتمالية التعثر المالي.


</div>


In [ ]:
df_FE['Debt_to_Income_Ratio'] = df_FE['Outstanding_Debt'] / (df_FE['Monthly_Inhand_Salary']*12) * 100

In [ ]:
px.box(df_FE, x="Debt_to_Income_Ratio", color="Credit_Score",  title="Distribution of Debt to Income Ratio by Credit Score")

In [ ]:
df_FE.drop(['Annual_Income'], axis=1, inplace=True)

<div dir="rtl">

* **قرار الهندسة (`Feature Creation & Selection`):** تم إنشاء ميزة **`Debt_to_Income_Ratio`** لقياس نسبة إجمالي الديون القائمة إلى الدخل السنوي، ثم حذف عمود الدخل السنوي (`Annual_Income`) لتطابقه شبه التام مع صافي الراتب الشهري (`Monthly_Inhand_Salary`) وتجنب تكرار المعلومات (`Multicollinearity`)، مع الإبقاء على باقي المتغيرات المالية المباشرة كما هي.

</div>


===================================================================================================================================================

<div dir="rtl">

### 3. طول تاريخ الائتمان (`Credit History Length` - 15%):

* **قرار الهندسة:** الاكتفاء بمتغير عمر التاريخ الائتماني (`Credit_History_Age`) كما هو بصيغته الرقمية (بالشهور) لوضوحه المباشر وتأثيره القوي في الفصل بين الفئات دون الحاجة لإضافة متغيرات مشتقة.

</div>


========================================================================================================================================

In [ ]:
df_FE.drop(['Num_of_Delayed_Payment',  'Type_of_Loan'], axis=1, inplace=True)

<div dir="rtl">

## ملخص مرحلة   (Feature Engineering Summary)

* **الميزات المستحدثة:** تم إنشاء مؤشر مخاطر السداد (**`Payment_History_Risk`**) وفق قاعدة الـ **30 يوماً** وعقوبة الحد الأدنى لتمثيل عامل تاريخ السداد (**35%**)، وإضافة نسبة الدين للدخل (**`Debt_to_Income_Ratio`**) لتمثيل عبء الديون (**30%**).
* **الميزات المحذوفة:** تم حذف (`Annual_Income`، `Num_of_Delayed_Payment`، `Payment_Behaviour`، `Type_of_Loan`) لتجنب التكرار الخطي وتقليل الأبعاد.
* **سبب الاكتفاء بباقي العوامل كما هي:** متغيرات عمر التاريخ الائتماني (**15%**)، وتنوع الائتمان (**10%**)، والاستعلامات الجديدة (**10%**) واضحة ومباشرة رقمياً، وإضافة معادلات مصطنعة لها قد يسبب تشتتاً للنموذج دون قيمة إضافية.

</div>


============================================================================================================================================================================

<h2 dir="ltr">Data Preprocessing for ML</h2>

<h3 dir="ltr">1) Separating Features and Target (X & y)</h3>

<div dir="rtl">

* **فصل المدخلات عن الهدف (`X` & `y`):** عزل عمود التصنيف الائتماني المستهدف (`Credit_Score`) في متغير (`y`)، ووضع باقي المتغيرات المستقلة في متغير المدخلات (`X`) مع استبعاد متغير العمر (`Age`) لعدم دخوله في معايير تقييم الجدارة الائتمانية (`FICO`) واستبعاد عمود الشهر (`Month`).

</div>


In [ ]:
X = df_FE.drop(columns=['Credit_Score', 'Age', 'Month'])
y = df_FE['Credit_Score']

<h3 dir="ltr">Why GroupShuffleSplit Instead of Standard train_test_split?</h3>

<div dir="rtl">

* **مشكلة التقسيم العادي (`Data Leakage`):** نظراً لأن كل عميل (`Customer_ID`) يمتلك **8 سجلات شهرية** في البيانات، فإن التقسيم العشوائي العادي سيضع بعض شهور العميل في التدريب وبعضها الآخر في الاختبار، مما يجعل النموذج يحفظ بيانات العميل الثابتة ويعطي دقة وهمية.
* **ميزة `GroupShuffleSplit`:** يتعامل هذا الأسلوب مع جميع سجلات العميل الواحد كمجموعة مغلقة؛ فيتم وضع العميل بالكامل إما في بيانات التدريب (`Train`) أو في بيانات الاختبار (`Test`)، مما يضمن اختبار النموذج على عملاء جدد تماماً، ثم يتم حذف عمود `Customer_ID` بعد إتمام التقسيم.

</div>


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=X['Customer_ID']))

X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
X_test, y_test = X.iloc[test_idx], y.iloc[test_idx]

X_train = X_train.drop(columns=['Customer_ID'])
X_test = X_test.drop(columns=['Customer_ID'])


<h3 dir="ltr">2) Create Numerical Pipelines</h3>

<div dir="rtl">

* **توحيد مقياس المتغيرات الرقمية (`RobustScaler`):** تم اختيار **RobustScaler** لتوحيد مقاييس المتغيرات الرقمية بدلاً من `StandardScaler`؛ نظراً لوجود قيم متطرفة (`Outliers`) طبيعية في البيانات المالية (مثل حجم الديون والأرصدة الشهرية)، حيث يعتمد `RobustScaler` على الوسيط (`Median`) والمدى الربيعي (`IQR`) مما يجعله مقاوماً لتأثير القيم الشاذة.

</div>


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler

num_colms = X_train.select_dtypes(include='number').columns

scaler = RobustScaler()
num_pipeline = Pipeline(steps=[('Scaler', scaler)])
num_pipeline


<h3 dir="ltr">3) Create Categorical Pipelines</h3>

<div dir="rtl">

* **ترميز المتغيرات النصية (`OneHotEncoder` & `BinaryEncoder`):**
  * تم تطبيق **`OneHotEncoder`** (مع `sparse_output=False` و `drop='first'`) على متغير **`Credit_Mix`** لتقليل التعدد الخطي (`Dummy Variable Trap`).
  * تم تطبيق **`BinaryEncoder`** على متغير **`Occupation`** لضغط فئاته الـ 15 في 4 أعمدة ثنائية فقط وتقليل الأبعاد (`Dimensionality`).

</div>


In [ ]:
from sklearn.preprocessing import OneHotEncoder
from category_encoders import BinaryEncoder

ohe_cols = ['Credit_Mix']
be_cols = ['Occupation']

ohe = OneHotEncoder(sparse_output=False, drop='first')
ohe_pipeline = Pipeline(steps=[('OHE', ohe)])

be = BinaryEncoder()
be_pipeline = Pipeline(steps=[('BE', be)])


<h3 dir="ltr">4) Assign Each Column to Corresponding Pipeline</h3>

<div dir="rtl">

* **دمج خطوط المعالجة (`ColumnTransformer`):** تم ربط خطوط معالجة المتغيرات الرقمية والنصية داخل أداة `ColumnTransformer` موحدة لتطبيق كل تحويل على الأعمدة المخصصة له بدقة، .

</div>


In [ ]:
from sklearn.compose import ColumnTransformer

preprocessing = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, num_colms),
        ('ohe', ohe_pipeline, ohe_cols),
        ('be', be_pipeline, be_cols)
    ]
)
preprocessing


### Select Best Model

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn .ensemble import RandomForestClassifier ,ExtraTreesClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score

le = LabelEncoder()

y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

models = [
    ('logisticregression',LogisticRegression()),
    ('KNN Classifier',KNeighborsClassifier()),
    ('Decisiontree classifier' , DecisionTreeClassifier(random_state=42)),
    ('Randomforest classifier' , RandomForestClassifier(random_state=42)),
    ('extratreesclassifier' , ExtraTreesClassifier(random_state=42)),
    ('Xgbclassifier' , XGBClassifier(random_state=42))
]


for name , model in models:
    pipe = Pipeline (steps=[
    ('preprocessing',preprocessing), 
    ('model' , model)
    ])

    pipe.fit(X_train, y_train_enc)

    y_train_pred = pipe.predict(X_train)
    y_test_pred = pipe.predict(X_test)

    

    train_f1 = f1_score(y_train_enc , y_train_pred , average='macro')*100
    test_f1 = f1_score(y_test_enc , y_test_pred, average='macro')*100

    print(f'Model : {name}')
    print("  ")
    print(f"Train Macro F1 {train_f1:.2f}%  |  Test Macro F1 {test_f1:.2f}%")
    print("  ")
    print("-" * 100)
    print("  ")






=====================================================================================================================================================================================================

<h3 dir="ltr">Baseline Models Comparison Summary</h3>

<div dir="rtl">

* **تحليل نتائج التجربة الأولية (`Baseline Results`):** بعد اختبار **6 خوارزميات تصنيف** باستخدام مقياس **`Macro F1-Score`** لضمان عدالة التقييم عبر جميع الفئات غير المتوازنة، تصدّر كلٌ من **`Random Forest`** (**67.56%**) و **`Extra Trees`** (**67.21%**) و **`XGBoost`** (**67.11%**) نتائج الاختبار (`Test Set`).
* **اكتشاف فرط التخصيص (`Overfitting Diagnosis`):** أظهر نموذج **`Random Forest`** دقة تدريب بلغت **100.00%** مقابل **67.56%** للاختبار، مما يشير إلى حفظ النموذج لتفاصيل بيانات التدريب نتيجة ترك الأشجار تنمو دون قيود، بينما أظهر **`XGBoost`** توازناً أفضل بفجوة أقل (**81.91%** للتدريب مقابل **67.11%** للاختبار).


</div>


# Random Forest

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

rf_pipe = Pipeline(steps=[
    ('preprocessing', preprocessing),
    ('model', RandomForestClassifier(class_weight='balanced', random_state=42 ))
])


param_dist = {
     'model__n_estimators':[100, 200, 300],
    'model__max_depth':[10,15,20],
    'model__min_samples_leaf': [2, 5, 10]
}

search = RandomizedSearchCV(
    estimator=rf_pipe,
    param_distributions=param_dist,
    n_iter=15,
    cv=5,
    scoring='f1_macro',
    random_state=42,
    n_jobs=2
)

search.fit(X_train, y_train_enc)

best_rf = search.best_estimator_
print("Best Parameters:", search.best_params_)

y_train_pred = best_rf.predict(X_train)
y_test_pred = best_rf.predict(X_test)

train_f1 = f1_score(y_train_enc, y_train_pred, average='macro') * 100
test_f1 = f1_score(y_test_enc, y_test_pred, average='macro') * 100

print(f"Tuned Random Forest -> Train Macro F1: {train_f1:.2f}% | Test Macro F1: {test_f1:.2f}%")


======================================================================================================================================================================================================================

<h3 dir="ltr">Random Forest Tuning Summary</h3>

<div dir="rtl">

* بعد ما عملنا Tuning لـ Random Forest باستخدام RandomizedSearchCV مع class_weight='balanced'، أحسن إعدادات طلعت هي `n_estimators=100` و `max_depth=20` و `min_samples_leaf=5`.
* التعديل ده فرق جداً في تقليل الـ Overfitting؛ سكور الـ Train نزل من **100.00%** لـ **82.28%**، وسكور الـ Test Macro F1 زاد من **67.56%** لـ **68.91%**.

</div>


=======================================================================================================================================================================================================

# XGBoost

In [ ]:
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier


sample_weights = compute_sample_weight(class_weight='balanced', y=y_train_enc)

xgb_pipe = Pipeline(steps=[
    ('preprocessing', preprocessing),
    ('model', XGBClassifier(random_state=42, n_jobs=2))
])


xgb_param_dist = {
    'model__n_estimators':[100, 200, 300],
    'model__max_depth': [10,15,20],
    'model__learning_rate': [0.03, 0.05, 0.1],
    'model__subsample': [0.8, 1.0],
    'model__colsample_bytree': [0.8, 1.0]
}


search_xgb = RandomizedSearchCV(
    estimator=xgb_pipe,
    param_distributions=xgb_param_dist,
    n_iter=15,
    cv=5,
    scoring='f1_macro',
    random_state=42,
    n_jobs=2
)
search_xgb.fit(X_train, y_train_enc, model__sample_weight=sample_weights)

 
 
best_xgb = search_xgb.best_estimator_
print("Best XGBoost Parameters:", search_xgb.best_params_)

y_train_pred_xgb = best_xgb.predict(X_train)
y_test_pred_xgb = best_xgb.predict(X_test)

train_f1_xgb = f1_score(y_train_enc, y_train_pred_xgb, average='macro') * 100
test_f1_xgb = f1_score(y_test_enc, y_test_pred_xgb, average='macro') * 100

print(f"Tuned XGBoost -> Train Macro F1: {train_f1_xgb:.2f}% | Test Macro F1: {test_f1_xgb:.2f}%")


<h3 dir="ltr">XGBoost Tuning Summary & Model Selection</h3>

<div dir="rtl">

* جربنا RandomizedSearchCV على XGBoost مع `sample_weight` عشان نعالج فرق النسب بين الفئات، وأحسن نتيجة طلعت مع `n_estimators=100` و `max_depth=10` و `learning_rate=0.05` و `subsample=0.8` و `colsample_bytree=0.8`.
* الموديل حقق **78.38%** في الـ Train و **68.90%** في الـ Test Macro F1، وهنعتمد **XGBoost** كموديل نهائي للمشروع لأن الفرق بين الـ Train والـ Test فيه أقل بكتير (**9.48%** بس مقابل **13.37%** في Random Forest)، وده معناه إنه مستقر أكتر ومش حافظ الداتا، غير إنه أخف وأسرع في مرحلة الـ Deployment.

</div>


================================================================================================================================================================================================================

<h3 dir="ltr">Tuned Models Comparison</h3>

<div dir="rtl">

| الموديل | Train Macro F1 | Test Macro F1 | الفرق (Overfitting Gap) | الملاحظات |
| :--- | :---: | :---: | :---: | :--- |
| **Tuned Random Forest** | 82.28% | **68.91%** | 13.37% | دقة عالية بس لسه فيه نسبة حفظ في الـ Train وحجم الموديل كبير. |
| **Tuned XGBoost (المختار)** | **78.38%** | **68.90%** | **9.48%** | نفس دقة الـ Test تقريباً بس استقرار أعلى بكتير وأسرع في الـ Deployment. |

</div>


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt


print(classification_report(y_test_enc, y_test_pred_xgb, target_names=le.classes_))


cm = confusion_matrix(y_test_enc, y_test_pred_xgb)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
disp.plot(cmap='Reds')
plt.title("Confusion Matrix - Tuned XGBoost")
plt.show()


<h3 dir="ltr">Final Model Evaluation Summary</h3>

<div dir="rtl">

* **تحليل نتائج التقييم النهائي:** أظهر تقرير التصنيف لـ **XGBoost** أداءً قوياً ومتوازناً عبر جميع الفئات؛ حيث سجلت فئة **Good** معدل استرجاع (`Recall`) وصل لـ **82%** (اصطاد **3,017** عميل من أصل **3,694**)، وحققت فئة **Poor** درجة **F1-Score** بلغت **71%**، بينما سجلت فئة **Standard** دقة (`Precision`) وصلت لـ **83%**.
* **قراءة الـ Confusion Matrix:** نجح الموديل في تصنيف **13,893** عميل بشكل صحيح من إجمالي **20,000** عميل في بيانات الاختبار، مع تركز الأخطاء المحدودة حول الفئة الوسطى (`Standard`) وندرة الخلط المباشر بين فئتي **Good** و **Poor**.

</div>


=========================================================================================================================================================================================

In [ ]:
import joblib


joblib.dump(best_xgb, "credit_score_pipeline.pkl")


joblib.dump(le, "label_encoder.pkl")
joblib.dump(best_xgb.named_steps['preprocessing'], "preprocessing_transformer.pkl")


joblib.dump(X_train.columns.tolist(), "inputs.pkl")

print("All files saved successfully!")


In [ ]:
X_train.columns.tolist()